# LAB-HW-09 — DDR Integrity / 第一次真实读写外部内存

**今天只新增一件事：** 在 benchmark AXI/burst 之前，先证明一个大块 deterministic payload 能稳定经过真实 KV260/K26 system-memory write/read roundtrip。

课程顺序：LAB-HW-08 之后。实际操作前置：LSN-016、LSN-017，以及已工作的 LAB-HW-05 PS/Linux boot path。

**Project Trace:** RMD-014 · T-HW-009/T-HW-011

本 Lab **不需要新 bitstream**。这是刻意的教学设计。

## 1. BRAM 和 DDR 解决的是不同问题

LAB-HW-07 使用片上 Block RAM（BRAM）：容量较小，但离 programmable logic 很近。

KV260 的 K26 system 提供 **4 GB DDR4** system memory。DDR 容量大得多，但它位于 processing-system memory path 后面，而不是 PL fabric 内部。

本 Lab 只问：**我们能否把 known bytes 稳定写进去，再完整读回来？**

这一章还不问：“PL AXI master 到底能搬多快？”

## 2. 先冻结这次到底在测哪一层

<svg xmlns="http://www.w3.org/2000/svg" width="980" height="310" viewBox="0 0 980 310" role="img" aria-label="LAB-HW-09 PS Linux managed DDR integrity path">
  <rect x="30" y="95" width="170" height="90" rx="10" fill="#eef0ff" stroke="#333"/>
  <text x="115" y="128" text-anchor="middle" font-size="14">Python helper</text>
  <text x="115" y="153" text-anchor="middle" font-size="12">PS / Linux</text>
  <rect x="250" y="95" width="180" height="90" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="340" y="128" text-anchor="middle" font-size="14">OS-managed</text>
  <text x="340" y="153" text-anchor="middle" font-size="12">anonymous mapping</text>
  <rect x="480" y="95" width="190" height="90" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="575" y="128" text-anchor="middle" font-size="14">PS memory path</text>
  <text x="575" y="153" text-anchor="middle" font-size="12">platform controller</text>
  <rect x="720" y="95" width="220" height="90" rx="10" fill="#fff3cd" stroke="#333"/>
  <text x="830" y="128" text-anchor="middle" font-size="14">K26 external DDR4</text>
  <text x="830" y="153" text-anchor="middle" font-size="12">4 GB system memory</text>
  <path d="M200 140 L250 140 M430 140 L480 140 M670 140 L720 140" stroke="#333" stroke-width="2"/>
  <polygon points="250,140 240,135 240,145" fill="#333"/><polygon points="480,140 470,135 470,145" fill="#333"/><polygon points="720,140 710,135 710,145" fill="#333"/>
  <rect x="480" y="225" width="220" height="55" rx="8" fill="#f4f4f4" stroke="#999" stroke-dasharray="7 5"/>
  <text x="590" y="258" text-anchor="middle" font-size="13">PL → AXI HP → DDR: LAB-HW-10</text>
</svg>

虚线那条路刻意留到下一章。

这里**不**对 DDR 使用 raw `/dev/mem`。allocation 由 Linux 管理，因此本 Lab 不会因为猜错 physical address 而覆盖 kernel、filesystem cache 或其他 process。

## 3. 冻结 integrity contract

physical mode 固定：

| 项目 | 数值 |
|---|---:|
| total test allocation | **64 MiB** |
| chunk size | **1 MiB** |
| chunks | 64 |
| access pattern | contiguous sequential chunks |
| payload | deterministic SHAKE256 chunk-index pattern |
| comparison | byte-for-byte + SHA-256 |
| root required | no |
| bitstream required | no |

timed write 前 helper 会先 **prefault** mapping，避免 first-touch page allocation 被偷偷混进 payload-copy timer。

payload generator 是 deterministic 的：同一个 helper、同一个 chunk index 会重新生成相同 bytes。

## 4. 任何 bandwidth 数字之前，先过 integrity

验收顺序强制为：

```console
write deterministic bytes
→ read every chunk
→ byte-for-byte compare
→ compare expected/observed SHA-256
→ INTEGRITY=PASS
→ only then print timing/bandwidth observations
```

哪怕只错一个 byte：

```console
INTEGRITY=FAIL
PERFORMANCE_BLOCKED=1
ERROR=DDR_INTEGRITY_MISMATCH
STATUS=FAIL
```

坏掉的数据路径不能靠一个“很快”的 timing 数字蒙混过关。

## 5. 上板前先 dry-run oracle

任意 development machine 上：

```bash
python boards/kv260/runtime/ddr_integrity.py \
  --dry-run \
  --json-out /tmp/lab-hw-09-dry-run.json
```

dry-run 使用 4 MiB，让 CI 保持快速，但仍执行同一套 deterministic generation、prefault、byte compare、SHA-256 compare 与“integrity first”逻辑。

最后应看到：

```console
INTEGRITY=PASS
PERFORMANCE_BLOCKED=0
BANDWIDTH_SCOPE=HOST_PATH_OBSERVATION_NOT_PEAK_DDR_OR_PL_AXI
STATUS=PASS
```

这只是 checker evidence，不是 physical T-HW-009 evidence。

## 6. 证明 corruption 真的会阻断 performance

仅用于 CI/test：

```bash
python boards/kv260/runtime/ddr_integrity.py \
  --dry-run \
  --inject-corruption-for-test
```

helper 会在 write 后翻转一个 byte。正确结果应该是 non-zero exit，并出现 `DDR_INTEGRITY_MISMATCH` 与 `PERFORMANCE_BLOCKED=1`。

physical mode 会拒绝 corruption option；不会让学生在真实实验里故意破坏这次测试数据。

## 7. 运行真实 KV260 memory sanity check

在 KV260 PS/Linux runtime host：

```bash
python3 /tmp/ddr_integrity.py \
  --physical \
  --json-out /tmp/lab-hw-09-trace.json
```

不需要 `sudo`。

physical mode 会先验证 `/proc/device-tree/model` 能识别 Kria/KV260 runtime，然后记录：

- board model；
- kernel / machine；
- OS identity；
- `MemTotal` / `MemAvailable`；
- swap totals；
- helper SHA-256；
- test geometry / access pattern。

如果 available memory 太低，不足以安全进行 64 MiB 测试，本 Lab 直接 FAIL，不在 memory pressure 下硬撑。

## 8. timing 数字到底表示什么？

integrity PASS 后 helper 会报告：

- `WRITE_ELAPSED_NS`
- `READ_ELAPSED_NS`
- `HOST_PATH_WRITE_MIB_PER_S`
- `HOST_PATH_READ_MIB_PER_S`

log 里会打印 timer boundary。

这些只是 **PS/Linux/userspace path 的 observation**，不是：

- peak DDR4 bandwidth；
- PL→DDR bandwidth；
- AXI burst efficiency；
- hardware-only latency；
- CPU-vs-FPGA benchmark。

cache 与 operating system 都属于这条 path。LAB-HW-10 才会引入受控的 PL/AXI measurement contract。

## 9. Failure class 必须分开

helper 区分：

- `NOT_KV260_RUNTIME_ENVIRONMENT`
- `INSUFFICIENT_AVAILABLE_MEMORY`
- `MEMORY_TEST_SETUP_FAILED`
- `DDR_INTEGRITY_MISMATCH`
- `INJECTION_NOT_ALLOWED_PHYSICAL`

runtime-environment failure 不是 DDR data mismatch；integrity mismatch 更不是 performance result。

integrity fail 时先 debug correctness，不开始调 access pattern。

## 10. 这一章不能证明什么

userspace write/read path 会经过 CPU cache。因此 LAB-HW-09 **不会**声称每个 timed load/store 都直接变成一次 electrical DDR transaction。

要证明更底层的事情，需要完全不同的 instrumentation/control boundary。

HW-09 证明的是更窄、但有价值的结论：在真实 KV260 PS/Linux system-memory path 上，一个较大的 deterministic working set 可以被分配、写入、读回并且不发生 corruption。

有了这个 correctness 基线，才有资格进入 HW-10。

## 11. Expected Evidence / Save Evidence

保留：

- `ddr_integrity.py` SHA-256；
- `lab-hw-09-trace.json`；
- 完整 terminal stdout；
- board model；
- kernel/OS identity；
- `MemTotal`、`MemAvailable`、swap snapshot；
- payload ID；
- 64 MiB total size / 1 MiB chunks；
- contiguous access-pattern label；
- expected / observed SHA-256；
- byte-compare PASS；
- timer-boundary string；
- raw write/read elapsed nanoseconds；
- host-path MiB/s observation；
- Git commit / experiment date。

**LAB-HW-09 不要求 bitstream hash**，因为本章刻意没有加入新的 PL design。

普通 CI dry-run 仍然只是 non-physical evidence。

## 12. Troubleshooting 顺序

1. 先确认自己确实在 KV260 PS/Linux runtime host。
2. 看 `MemAvailable`；必要时关闭占内存较大的应用。
3. 重新跑 dry-run，确认 oracle 自己仍然 PASS。
4. 再跑 physical mode，比对 expected/observed SHA-256。
5. 如果 mismatch 可重复，先保存 JSON/terminal evidence，再改东西。
6. integrity 没稳定之前，不进入 AXI/burst tuning。

## 13. Human Check

请解释：

1. 为什么 HW-09 使用 Linux-managed memory，而不是猜一个 `/dev/mem` DDR address？
2. 为什么 64 MiB 远大于 HW-07 的 4 KiB BRAM teaching window？
3. 为什么同时做 byte-for-byte 与 SHA-256 compare？
4. 为什么错一个 byte 就必须阻断所有 performance conclusion？
5. 为什么 `HOST_PATH_*_MIB_PER_S` 不是 peak DDR bandwidth？
6. 为什么本章不需要新 bitstream？
7. LAB-HW-10 刻意延后了哪个新概念？

## 14. Engineering handoff

HW-09 在不引入新 PL protocol 问题的情况下，建立 external system memory correctness baseline。

进入 HW-10 之前，**先停在这里完成独立的 Ubuntu / u-dma-buf prerequisite**：

`docs/zh/KV260_UDMABUF_SETUP.md`

并在 KV260 runtime host 上通过：

```bash
sudo python3 /tmp/preflight_udmabuf.py \
  --physical \
  --json-out /tmp/lab-hw-10-udmabuf-preflight.json
```

只有 prerequisite `STATUS=PASS`，再进入：

```console
LAB-HW-09
OS-managed DDR integrity
        ↓
Ubuntu/u-dma-buf setup + preflight
        ↓
LAB-HW-10
PL → AXI high-performance port → DDR
small/scattered vs contiguous/burst measurement
```

HW-10 可以开始改变 access path，但必须保留本章规则：**数据一旦 corrupt，performance claim 全部无效。**


## 15. 官方依据

- AMD Kria KV260 Vision AI Starter Kit Data Sheet（DS986）Product Details：KV260 提供 4 GB non-ECC DDR4 system memory。
- AMD Zynq UltraScale+ MPSoC Processing System Product Guide（PG201）Slave Interface：PL 的 high-performance `S_AXI_HP0..3_FPD` interface 提供通往 FPD main switch / DDR 的 non-coherent PL path；这条 PL path 刻意留到 LAB-HW-10。
- 仓库教学事实源：`lessons/zh/17_external_memory_ddr.ipynb` 明确要求 DDR hello-world 先证明 integrity，再做 bandwidth benchmark。

AMD 官方参考：
- https://docs.amd.com/r/en-US/ds986-kv260-starter-kit/Product-Details
- https://docs.amd.com/r/en-US/pg201-zynq-ultrascale-plus-processing-system/Slave-Interface